### For testing

In [92]:
#import libraries
import pandas as pd
import numpy as np
from scipy.signal import savgol_filter

# import other functions
from app.pcr.sampleid_mapping import mapping_sampleid
from app.kits.selected_kit import load_selected_kit

In [93]:
file = "C:\\Users\\anett\\OneDrive\\Dokumentumok\\SE-EMK\\III.felev\\Szakdolgozat\\EDS_file\\COVID\\CIRM_5516_5517.eds"
selected_kit_name = "Respiratory Panel (SarsCov2 + Influenza)"
sample_ids = "C:\\Users\\anett\\OneDrive\\Dokumentumok\\SE-EMK\\III.felev\\Szakdolgozat\\EDS_file\\COVID\\CIRM_5516_5517_sample_id.xlsx"

sampleid_df = pd.read_excel(sample_ids, sheet_name="Sample_ID")
print(sampleid_df.head())

window=9
poly=2

   well  well_position    sample_id
0      1            A1  111749975.0
1      2            A2          NaN
2      3            A3  111208676.0
3      4            A4          NaN
4      5            A5  111869250.0


In [139]:

results = []
df = mapping_sampleid(file, sampleid_df)
df_final = df[df['cycle'] > 10]
channels = load_selected_kit(selected_kit_name)[2]

x = df_final['cycle'].values
abs_min_derivative_map = {
        "FAM": 6000, "VIC": 942, "ABY": 3000, "Cy5": 2583, "ROX": 8055 
    } 

for (well, well_position, sid), df_sample in df_final.groupby(['well', 'well_position', 'sample_id']):
    row_result = {"well": well, "well_position": well_position, 'sample_id': sid}
        
    
    channel_data = {}
    for ch in channels:
        if ch not in df_sample.columns: continue
        y = df_sample[ch].astype(float).values
        y_smooth = savgol_filter(y, window, poly)
        dy = savgol_filter(y_smooth, window, poly, deriv=1)
        x = df_final['cycle'].values
        dy_max = dy.max()
        abs_min = abs_min_derivative_map.get(ch, 3000)

        if dy_max < abs_min:
                row_result[ch] = "negatív"
                continue
    
        else: 
            d2y = savgol_filter(y_smooth, window, poly, deriv=2)
            mask = (dy > 0)
            
            if not np.any(mask):
                row_result[ch] = "negatív"
                continue

            valid_d2 = d2y[mask]
            #valid_x = x[mask]
            
            ct_idx_local = np.argmax(valid_d2)
            row_result[ch] = float(x[ct_idx_local])

    results.append(row_result)

results_df = pd.DataFrame(results)

results_df.to_excel("C:\\Users\\anett\\Downloads\\results_02.xlsx", index=False)



In [140]:
def evaluate_PCR_curves(
        file, 
        sampleid_df,
        selected_kit_name, 
        window=9,
        poly=2):
    
    results = []
    df = mapping_sampleid(file, sampleid_df)
    df_final = df[df['cycle'] > 10]
    channels = load_selected_kit(selected_kit_name)[2]
    x = df_final['cycle'].values

    # Alapmeredekség beállítása csatornánként
    abs_min_derivative_map = {
        "FAM": 6000, "VIC": 2500, "ABY": 3000, "Cy5": 3000, "ROX": 8000 } 

    # c
    for (well, well_position, sid), df_sample in df_final.groupby(['well', 'well_position', 'sample_id']):
        row_result = {"well": well, "well_position": well_position, 'sample_id': sid}
        
        for ch in channels:
            if ch not in df_sample.columns: continue
            y = df_sample[ch].astype(float).values
            y_smooth = savgol_filter(y, window, poly)
            dy = savgol_filter(y_smooth, window, poly, deriv=1)
            x = df_final['cycle'].values
            dy_max = dy.max()
            abs_min = abs_min_derivative_map.get(ch, 3000)

            if dy_max < abs_min:
                row_result[ch] = "negatív"
                continue
    
            else: 
                d2y = savgol_filter(y_smooth, window, poly, deriv=2)
                mask = (dy > 0)
            
                if not np.any(mask):
                    row_result[ch] = "negatív"
                    continue

                valid_d2 = d2y[mask]
            
                ct_idx_local = np.argmax(valid_d2)
                row_result[ch] = float(x[ct_idx_local])

        results.append(row_result)

    results_df = pd.DataFrame(results)
    evaluate_PCR_curves_result = results_df.melt(
        id_vars=["well", "well_position", 'sample_id'],
        var_name="dye",
        value_name="Result")

    evaluate_PCR_curves_result = evaluate_PCR_curves_result.sort_values(by="well").reset_index(drop=True)

    return evaluate_PCR_curves_result

### Egy well dy_max értékei

In [154]:
#import libraries
import pandas as pd
import numpy as np
from scipy.signal import savgol_filter

# import other functions
from app.pcr.sampleid_mapping import mapping_sampleid
from app.kits.selected_kit import load_selected_kit

file_2 = "C:\\Users\\anett\\OneDrive\\Dokumentumok\\SE-EMK\\III.felev\\Szakdolgozat\\EDS_file\\COVID\\CIRM_ 08cd_5498 .eds"
file_3 = "C:\\Users\\anett\\OneDrive\\Dokumentumok\\SE-EMK\\III.felev\\Szakdolgozat\\EDS_file\\COVID\\CIRM_43ec_5488.eds"
selected_kit_name = "Respiratory Panel (SarsCov2 + Influenza)"
sample_ids_2 = "C:\\Users\\anett\\OneDrive\\Dokumentumok\\SE-EMK\\III.felev\\Szakdolgozat\\EDS_file\\COVID\\CIRM_ 08cd_5498_sample_id.xlsx"
sample_ids_3 = "C:\\Users\\anett\\OneDrive\\Dokumentumok\\SE-EMK\\III.felev\\Szakdolgozat\\EDS_file\\COVID\\CIRM_43ec_5488_sample_id.xlsx"
sampleid_df = pd.read_excel(sample_ids_2, sheet_name="Sample_ID")


df = mapping_sampleid(file_3, sampleid_df_3)
df_final = df[df['cycle'] > 10]
selected_well = 'C15'
df_final_selected = df_final[df_final['well_position'] == selected_well]
channels = load_selected_kit(selected_kit_name)[2]

result = {}
for ch in channels:
    y = df_final_selected[ch].astype(float).values
    y_smooth = savgol_filter(y, window, poly)
    dy = savgol_filter(y_smooth, window, poly, deriv=1)
    dy_max = dy.max()
    result[ch] = dy_max

print(result)




{'FAM': np.float64(2976.549117753586), 'VIC': np.float64(6906.798331962472), 'ROX': np.float64(2565.9777798401283), 'CY5': np.float64(3331.4684134672307)}


### Artifactumok (görbe együtt mozgások kiszűrése)

In [174]:
sample_ids_3 = "C:\\Users\\anett\\OneDrive\\Dokumentumok\\SE-EMK\\III.felev\\Szakdolgozat\\EDS_file\\COVID\\CIRM_43ec_5488_sample_id.xlsx"
sampleid_df = pd.read_excel(sample_ids_2, sheet_name="Sample_ID")
selected_kit_name = "Respiratory Panel (SarsCov2 + Influenza)"
df = mapping_sampleid(file_3, sampleid_df_3)

selected_well = 'I3'
df_sample = df[df['well_position'] == selected_well]
cycle_col="cycle"
min_cycle=1
max_cycle=23
window=9
poly=2
local_span=2         # +/- 2 pont, tehát 5 pontos lokális ablak
corr_threshold=0.90
min_corr_threshold=0.90
min_channels=2

df_early = df_sample[
        (df_sample[cycle_col] >= min_cycle) & (df_sample[cycle_col] <= max_cycle)
    ].copy()



x = df_early[cycle_col].astype(float).values
deriv_dict = {}

# 1. deriváltak számítása csatornánként
for ch in channels:
    y = df_early[ch].astype(float).values
    y_smooth = savgol_filter(y, window, poly)
    dy = savgol_filter(y_smooth, window, poly, deriv=1)
    deriv_dict[ch] = dy


used_channels = list(deriv_dict.keys())

valid_windows = []

for i in range(len(x) - 2):
    idx = slice(i, i + 3)

    local_derivs = []
    for ch in used_channels:
        local_derivs.append(deriv_dict[ch][idx])

    deriv_matrix = np.vstack(local_derivs)
    corr_matrix = np.corrcoef(deriv_matrix)

    n = corr_matrix.shape[0]
    upper_vals = corr_matrix[np.triu_indices(n, k=1)]

    if len(upper_vals) == 0:
        continue

    mean_corr = np.nanmean(upper_vals)
    min_corr = np.nanmin(upper_vals)

    if mean_corr >= corr_threshold and min_corr >= min_corr_threshold:
        # az ablak "közepét" tekintjük reprezentatív ciklusnak
        valid_windows.append(int(x[i + 1]))


 # 3. összefüggő blokk keresése
valid_windows = sorted(set(valid_windows))

blocks = []
block = [valid_windows[0]]

for v in valid_windows[1:]:
    if v == block[-1] + 1:
        block.append(v)
    else:
        blocks.append(block)
        block = [v]

blocks.append(block)

best_block = max(blocks, key=len)


    # 4. utolsó együttmozgó ciklus + 1
print(best_block[-1] + 1)


16


### Dy_max tesztelés csatornánként, futásonként

In [135]:
file_2 = "C:\\Users\\anett\\OneDrive\\Dokumentumok\\SE-EMK\\III.felev\\Szakdolgozat\\EDS_file\\COVID\\CIRM_ 08cd_5498 .eds"
selected_kit_name = "Respiratory Panel (SarsCov2 + Influenza)"
sample_ids_2 = "C:\\Users\\anett\\OneDrive\\Dokumentumok\\SE-EMK\\III.felev\\Szakdolgozat\\EDS_file\\COVID\\CIRM_ 08cd_5498_sample_id.xlsx"

sampleid_df_2 = pd.read_excel(sample_ids_2, sheet_name="Sample_ID")
df_2 = mapping_sampleid(file_2, sampleid_df_2)
df_final_2 = df_2[df_2['cycle'] > 10]

result_2 = {}
for ch in channels:
    dy_max_list= []
    for well_position, df_sample in df_final_2.groupby('well_position'):
        y = df_sample[ch].astype(float).values
        y_smooth = savgol_filter(y, window, poly)
        dy = savgol_filter(y_smooth, window, poly, deriv=1)
        dy_max = dy.max()
        dy_max_list.append(dy_max)
    result_2[ch] = max(dy_max_list)

print(result_2)



{'FAM': np.float64(62576.87923376614), 'VIC': np.float64(8876.934042712834), 'ROX': np.float64(78426.03804617595), 'CY5': np.float64(19379.88211038959)}


In [137]:
file_3 = "C:\\Users\\anett\\OneDrive\\Dokumentumok\\SE-EMK\\III.felev\\Szakdolgozat\\EDS_file\\COVID\\CIRM_9af8_5638.eds"
selected_kit_name = "Respiratory Panel (SarsCov2 + Influenza)"
sample_ids_3 = "C:\\Users\\anett\\OneDrive\\Dokumentumok\\SE-EMK\\III.felev\\Szakdolgozat\\EDS_file\\COVID\\CIRM_9af8_5638_sample_id.xlsx"

sampleid_df_3 = pd.read_excel(sample_ids_3, sheet_name="Sample_ID")
df_3 = mapping_sampleid(file_3, sampleid_df_3)
df_final_3 = df_3[df_3['cycle'] > 10]

result_3 = {}
for ch in channels:
    dy_max_list= []
    for well_position, df_sample in df_final_3.groupby('well_position'):
        y = df_sample[ch].astype(float).values
        y_smooth = savgol_filter(y, window, poly)
        dy = savgol_filter(y_smooth, window, poly, deriv=1)
        dy_max = dy.max()
        dy_max_list.append(dy_max)
    result_3[ch] = max(dy_max_list)

print(result_3)


{'FAM': np.float64(76906.09733441549), 'VIC': np.float64(7937.079188961031), 'ROX': np.float64(47485.419896103835), 'CY5': np.float64(26561.47439841267)}


In [138]:
file_4 = "C:\\Users\\anett\\OneDrive\\Dokumentumok\\SE-EMK\\III.felev\\Szakdolgozat\\EDS_file\\COVID\\CIRM_43ec_5488.eds"
selected_kit_name = "Respiratory Panel (SarsCov2 + Influenza)"
sample_ids_4 = "C:\\Users\\anett\\OneDrive\\Dokumentumok\\SE-EMK\\III.felev\\Szakdolgozat\\EDS_file\\COVID\\CIRM_43ec_5488_sample_id.xlsx"

sampleid_df_4 = pd.read_excel(sample_ids_4, sheet_name="Sample_ID")
df_4 = mapping_sampleid(file_4, sampleid_df_4)
df_final_4 = df_4[df_4['cycle'] > 10]

result_4 = {}
for ch in channels:
    dy_max_list= []
    for well_position, df_sample in df_final_4.groupby('well_position'):
        y = df_sample[ch].astype(float).values
        y_smooth = savgol_filter(y, window, poly)
        dy = savgol_filter(y_smooth, window, poly, deriv=1)
        dy_max = dy.max()
        dy_max_list.append(dy_max)
    result_4[ch] = max(dy_max_list)

print(result_4)

{'FAM': np.float64(58464.73024927844), 'VIC': np.float64(8416.200803751794), 'ROX': np.float64(38289.842710750316), 'CY5': np.float64(27686.22399163056)}


Well_position vs well

In [4]:
def well_position_to_well(well_position: str) -> int:
    # Szétbontás (pl. "M6" → "M" és "6")
    row_letter = well_position[0].upper()
    col_number = int(well_position[1:])
    
    # Sor index meghatározása (A=0, B=1, ..., P=15)
    row_index = ord(row_letter) - ord('A')
    
    # Validáció
    if row_index < 0 or row_index > 15:
        raise ValueError("Érvénytelen sor (A-P lehet)")
    if col_number < 1 or col_number > 24:
        raise ValueError("Érvénytelen oszlop (1-24 lehet)")
    
    # Well sorszám kiszámítása
    well = row_index * 24 + col_number
    
    return well

print(well_position_to_well("P28"))  # Példa használat

ValueError: Érvénytelen oszlop (1-24 lehet)